# AML Graph Autoencoder - End-to-End Pipeline with Ground Truth Verification
This notebook demonstrates the unified AML pipeline: Simulation -> Graph Construction -> GATv2 Training -> Anomaly Detection Verification.

In [ ]:
import os, sys
sys.path.append(os.path.abspath("../"))
import yaml, torch, pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from src.utils.reproducibility import set_seed, save_checkpoint, load_checkpoint, get_device
from src.data.generator import generate_transactions
from src.graph.builder import GraphBuilder
from src.models.gnn import AMLGraphAutoencoder, compute_combined_loss
from src.explain.interpreter import AnomalyInterpreter

# 0. Setup
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
set_seed(42)
device = get_device(config)

DATA_PATH = '../data/ledger.csv'
USE_EXISTING_DATA = False

os.makedirs('../artifacts', exist_ok=True)
os.makedirs('../output', exist_ok=True)
os.makedirs('../data', exist_ok=True)

# 1. Data Generation & Ground Truth Labels
NUM_CUSTOMERS = config['data']['num_customers']
full_tx_df = generate_transactions(num_customers=NUM_CUSTOMERS, num_days=40, anomaly_ratio=0.05)
c_id_col = config['data']['column_mapping']['customer_id']

# IDENTIFY NEWLY DEFINED ANOMALY CUSTOMERS
pt_customers = full_tx_df[full_tx_df['label'] == 1][c_id_col].unique().tolist()
fanin_customers = full_tx_df[full_tx_df['label'] == 2][c_id_col].unique().tolist()
repeat_customers = full_tx_df[full_tx_df['label'] == 3][c_id_col].unique().tolist()
self_customers = full_tx_df[full_tx_df['label'] == 4][c_id_col].unique().tolist()
persistent_customers = full_tx_df[full_tx_df['label'] == 5][c_id_col].unique().tolist()

print(f"--- Ground Truth Anomaly IDs ---")
print(f"Pass-Through Customers: {pt_customers}")
print(f"Persistent Link Customers (5 daily matches): {persistent_customers}")
print(f"Repeated Layering Customers: {repeat_customers}")
print(f"Self-Passthrough Customers: {self_customers}")

train_df = full_tx_df[full_tx_df['date'] < '2026-02-01'].copy()
oot_df = full_tx_df[full_tx_df['date'] >= '2026-02-01'].copy()

# 2. Build Training Graph
builder = GraphBuilder(config)
train_data = builder.build_graph(train_df)

# 3. Preprocessing
node_scaler = StandardScaler()
edge_scaler = StandardScaler()
x_scaled = node_scaler.fit_transform(train_data.x.numpy())
edge_attr_scaled = edge_scaler.fit_transform(train_data.edge_attr.numpy())

train_data.x = torch.from_numpy(x_scaled).float().to(device)
train_data.edge_index = train_data.edge_index.to(device)
train_data.edge_attr = torch.from_numpy(edge_attr_scaled).float().to(device)

# 4. Training
model = AMLGraphAutoencoder(train_data.num_node_features, train_data.num_edge_features, config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(101):
    optimizer.zero_grad()
    z, x_recon, edge_recon = model(train_data.x, train_data.edge_index, train_data.edge_attr)
    loss, _, _ = compute_combined_loss(train_data.x, x_recon, train_data.edge_attr, edge_recon, config)
    loss.backward()
    optimizer.step()

# 5. OOT Inference
oot_data = builder.build_graph(oot_df)
oot_data.x = torch.from_numpy(node_scaler.transform(oot_data.x.numpy())).float().to(device)
oot_data.edge_attr = torch.from_numpy(edge_scaler.transform(oot_data.edge_attr.numpy())).float().to(device)
oot_data.edge_index = oot_data.edge_index.to(device)

model.eval()
with torch.no_grad():
    _, x_recon_oot, edge_recon_oot = model(oot_data.x, oot_data.edge_index, oot_data.edge_attr)
    node_errors = torch.mean((oot_data.x - x_recon_oot)**2, dim=1)
    edge_errors = torch.mean((oot_data.edge_attr - edge_recon_oot)**2, dim=1)

# 6. Verification of Persistent Link
print("\n--- Verification: Persistent Link (Label 5) ---")
for p_id in persistent_customers[:2]:
    internal_idx = builder.customer_map.get(p_id)
    # Check if this customer is a source in any edge
    edges = oot_data.collapsed_edges_df
    relevant_edges = edges[edges['source_id'] == p_id]
    if not relevant_edges.empty:
        print(f"Customer {p_id} Edge Stats:")
        display(relevant_edges[['source_id', 'target_id', 'edge_frequency', 'total_inferred_amt']])
    else:
        print(f"Customer {p_id}: No matching edges found in OOT period.")

# 7. Interpretation & Export
interpreter = AnomalyInterpreter(model, config)
interpreter.save_anomalies_to_excel(oot_data, node_errors, edge_errors, oot_df, '../output/anomalies.xlsx', inv_map=builder.inv_customer_map)
